In [0]:
dbutils.widgets.text("secret_scope", "eventhub")
dbutils.widgets.text("secret_key", "eh-connection-string")
dbutils.widgets.text("eh_name", "roksolana-wikipedia-recentchange")
dbutils.widgets.text("max_events", "2000")
dbutils.widgets.text("wiki_filter", "en.wikipedia.org")

SECRET_SCOPE = dbutils.widgets.get("secret_scope")
SECRET_KEY = dbutils.widgets.get("secret_key")
EH_NAME = dbutils.widgets.get("eh_name")
MAX_EVENTS = int(dbutils.widgets.get("max_events"))
WIKI_FILTER = dbutils.widgets.get("wiki_filter")

EH_CONN_STR = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

In [0]:
import json
import requests
import sseclient
from azure.eventhub import EventData
from azure.eventhub.aio import EventHubProducerClient

FIELDS = ["id", "type", "title", "user", "bot", "minor", "timestamp", "wiki", "server_name", "length"]

headers = {
    "User-Agent": "DatabricksLab3Producer/1.0 (roksolana.shendiu770@softserve.academy)"
}


async def run():
    producer = EventHubProducerClient.from_connection_string(
        conn_str=EH_CONN_STR,
        eventhub_name=EH_NAME,
    )

    response = requests.get(
        "https://stream.wikimedia.org/v2/stream/recentchange",
        stream=True,
        headers=headers,
        timeout=(10, 60),
    )

    sent_count = 0

    async with producer:
        batch = await producer.create_batch()
        try:
            client = sseclient.SSEClient(response)
            for event in client.events():
                if not event.data:
                    continue
                record = json.loads(event.data)
                if record.get("server_name") != WIKI_FILTER:
                    continue
                payload = {k: record.get(k) for k in FIELDS}
                try:
                    batch.add(EventData(json.dumps(payload)))
                except ValueError:
                    await producer.send_batch(batch)
                    batch = await producer.create_batch()
                    batch.add(EventData(json.dumps(payload)))
                sent_count += 1
                if sent_count >= MAX_EVENTS:
                    break
            if len(batch) > 0:
                await producer.send_batch(batch)
        finally:
            response.close()


await run()